# Stage 3: evaluate

**Goal.** Predict the held-out scenes and produce one row per 256 px tile: how wrong the model
was there, and how confident it was. That table (`tiles.csv`) is the input of the analysis.

**Command-line equivalent:** `p037 evaluate --config <config>`

**Code:** `src/p037/evaluate.py`, `src/p037/metrics.py`

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from p037 import plots                      # sets the figure style used in the report
from p037.config import REPO_ROOT, load_config
from p037.datasets import get_dataset

# The experiment this notebook works on. smoke.yaml is generated data: it runs anywhere in minutes
# and its numbers mean nothing. On BluePebble use analysis/p037/configs/exp_b_usf.yaml.
# (Setting the environment variable P037_CONFIG before starting Jupyter does the same for all notebooks.)
CONFIG = os.environ.get("P037_CONFIG", "analysis/p037/configs/smoke.yaml")
cfg = load_config(REPO_ROOT / CONFIG)
ds = get_dataset(cfg)
DRY_RUN = cfg.dataset == "synthetic"
print(f"experiment {cfg.name!r} on dataset {cfg.dataset!r}")
print("dataset files:", cfg.dataset_dir)
print("run folder:   ", cfg.out)

## 3.1 Load the model

A trained experiment uses its own best checkpoint (`best_f1.pt`). An experiment with
`init: pretrained` and no checkpoint uses the published weights.

In [ ]:
import rasterio
import torch
from rasterio.windows import Window

from p037.datasets.base import INVALID, uint8_to_db
from p037.evaluate import Predictor
from p037.model import load_for_eval, pick_device

device = pick_device()
model, origin = load_for_eval(cfg, device)
meta = ds.meta()
predictor = Predictor(cfg, model, meta.channels, device)
print("weights:", origin)
print("flood rule:", cfg.flood_rule)

## 3.2 One window, step by step

A scene is far too large for one forward pass, so it is predicted in windows. A pixel at the edge
of a window is predicted with less surrounding context than one in the middle, so each window is
read with a margin (`crop`) on every side and the margin's predictions are thrown away.

**Confidence.** The prediction that is scored comes from one pass with dropout switched off.
Then the same window is predicted `mc_passes` more times with dropout **on**: each pass removes a
different random part of the network. Where the passes disagree, the model is unsure. The variance
across passes is the per-pixel uncertainty (MC dropout).

In [ ]:
events = [e for e in (cfg.eval_events or ds.test_events) if ds.has_scene(e)]
event = events[0]
f = ds.scene_files(event)
W, C = cfg.window, cfg.crop

with rasterio.open(f["image"]) as im, rasterio.open(f["gt"]) as gt, rasterio.open(f["valid"]) as va:
    # pick the window of the tile grid that holds the most labelled flood
    step = max(1, W // 64)
    small = gt.read(1, out_shape=(max(1, gt.height // step), max(1, gt.width // step)))
    rows, cols = -(-gt.height // W), -(-gt.width // W)
    counts = np.zeros((rows, cols))
    for r in range(rows):
        for c in range(cols):
            block = small[r * W // step:(r + 1) * W // step, c * W // step:(c + 1) * W // step]
            counts[r, c] = np.isin(block, list(meta.classes)).sum()
    r, c = np.unravel_index(counts.argmax(), counts.shape)
    core = Window(c * W, r * W, W, W)
    with_margin = Window(c * W - C, r * W - C, W + 2 * C, W + 2 * C)
    img = im.read(window=with_margin, boundless=True, fill_value=250)      # 250 = 0 dB, the model's padding value
    label = gt.read(1, window=core, boundless=True, fill_value=INVALID)
    valid = va.read(1, window=core, boundless=True, fill_value=0).astype(bool)

out = predictor(img[None], crop=C)                      # arrays of the core only: the margin is cropped
print("input with margin:", img.shape, "-> prediction of the core:", out["prob"][0].shape)

In [ ]:
post = meta.channels.index(cfg.post_channels[0]) if cfg.post_channels[0] in meta.channels else 0
panels = [("image, " + meta.channels[post] + " (dB)", uint8_to_db(img[post, C:C + W, C:C + W]), "gray", (-25, 0)),
          ("label", np.ma.masked_equal(label, INVALID), plots.SEQ, (0, max(meta.classes))),
          ("probability of flood", out["prob"][0], plots.SEQ, (0, 1)),
          (f"prediction (threshold {cfg.threshold})", out["pred"][0], plots.SEQ, (0, 1))]
if "mc_var" in out:
    panels.append(("MC-dropout variance", out["mc_var"][0], plots.SEQ, (0, np.percentile(out["mc_var"][0], 99) + 1e-9)))
fig, axes = plt.subplots(1, len(panels), figsize=(3.2 * len(panels), 3.5))
for ax, (title, data, cmap, (lo, hi)) in zip(axes, panels):
    ax.imshow(data, cmap=cmap, vmin=lo, vmax=hi, interpolation="nearest")
    ax.set_title(title, loc="left"); ax.set_xticks([]); ax.set_yticks([]); ax.grid(visible=False)
fig.suptitle(f"{event}, window row {r}, column {c}", x=0.01, ha="left")
fig.tight_layout()

## 3.3 From pixels to a tile's score

Each tile gets four counts against the label: true positives (flood found), false positives
(flood claimed, none there), false negatives (flood missed) and true negatives. Everything else,
F1, precision, recall, follows from the counts.

Scoring is **one-vs-rest per class**: when open flood is scored, urban-flood pixels are left out
rather than counted as false alarms, because the model has a single flood output.

In [ ]:
from p037.metrics import class_sets, confusion, scores

rows = []
for name, (positive, exclude) in class_sets(meta.classes).items():
    tp, fp, fn, tn = confusion(out["pred"][0], label, valid, positive, exclude)
    rows.append({"class": name, "tp": tp, "fp": fp, "fn": fn, "tn": tn, **scores(tp, fp, fn, tn)})
pd.DataFrame(rows).set_index("class").round(3)

## 3.4 All scenes

`evaluate_scenes` does the above for every window of every test scene and writes `tiles.csv`.
Scenes already scored are reused. With 1 + 20 passes per window this needs a GPU for real scenes,
so on BluePebble submit `evaluate.sbatch`; the cell runs here for the dry run or when a GPU is present.

In [ ]:
from p037.evaluate import evaluate_scenes

tiles_path = cfg.out / "tiles.csv"
if DRY_RUN or device == "cuda":
    tiles_path = evaluate_scenes(cfg, device)
elif not tiles_path.exists():
    print("Submit on BluePebble:  sbatch analysis/p037/slurm/evaluate.sbatch", CONFIG)

if tiles_path.exists():
    tiles = pd.read_csv(tiles_path)
    print(len(tiles), "tiles;", len(tiles.columns), "columns")
    display(pd.read_csv(cfg.out / "tiles_summary.csv").round(3))

`tiles_summary.csv` pools the pixel counts of all tiles of a scene before computing the scores.
Those are the numbers to quote for a scene. An average of per-tile F1 is a different quantity and
is dominated by tiles with very little flood.

The columns of `tiles.csv`:

| Columns | Meaning |
|---|---|
| `tile_id`, `event`, `tile_row`, `tile_col`, `lon`, `lat` | which tile, and where its centre is |
| `n_valid`, `n_gt_<class>` | valid pixels and labelled pixels per class |
| `<target>_<class>_tp/fp/fn/tn` and `_f1/_iou/_precision/_recall` | counts and scores, per class and for `any` |
| `frac_<target>`, `mean_prob_<target>` | share of the tile predicted as flood, mean probability |
| `mc_var_<target>`, `mean_entropy_<target>` | MC-dropout uncertainty, averaged over the tile |

## 3.5 Where are the errors, and where is the model unsure?

One cell per tile, laid out as in the scene. If uncertainty were a perfect guide to error the two
maps would look the same. Where they differ, the model is wrong without knowing it, which is the
case the landscape proxies are meant to help with.

In [ ]:
if tiles_path.exists():
    pre = f"{meta.target}_{cfg.score_class}"
    n = tiles[[f"{pre}_{k}" for k in ("tp", "fp", "fn", "tn")]].sum(axis=1)
    tiles["err_rate"] = (tiles[f"{pre}_fp"] + tiles[f"{pre}_fn"]) / n.where(n > 0)
    tiles["log_mc_var"] = np.log10(tiles[f"mc_var_{meta.target}"] + 1e-12)
    for event in sorted(tiles["event"].unique()):
        d = tiles[tiles["event"] == event]
        fig, axes = plt.subplots(1, 2, figsize=(9, 4))
        for ax, col, title in zip(axes, ("err_rate", "log_mc_var"), ("error rate", "log10 MC-dropout variance")):
            data = d.pivot(index="tile_row", columns="tile_col", values=col).to_numpy()   # the scene's tile grid
            lo, hi = np.nanpercentile(data, [2, 98])
            ax.imshow(data, cmap=plots.SEQ, vmin=lo, vmax=max(hi, lo + 1e-9), interpolation="nearest")
            ax.set_title(f"{plots.nice_event(event)}: {title}", loc="left")
            ax.set_xticks([]); ax.set_yticks([]); ax.grid(visible=False)
        fig.tight_layout()

## Check before moving on

- The pooled scores per scene are in the range you expect, and the tile count matches the scene
  size (rows x columns of 256 px tiles).
- The probability map of section 3.2 looks like the label, not like noise or like the raw image.

Next: `04_proxies.ipynb`.